[Guide Index](README.md)


# 04c SHACL Rules

`StarShaclSchema.apply_rules()` executes SHACL's rule-based inference (`sh:rule`, `sh:TripleRule`, `sh:SPARQLRule`) - a *third* processing mode alongside `validate()` (checks conformance, never mutates) and `evaluate()` (computes `sh:values`-declared virtual properties on demand, never persists). Rules are different from both: they **materialize new, real triples** into the data graph, computed from existing ones. (A related but separate utility, `extract_subgraph()`, pulls out exactly the triples a shape covers for one focus node - a plain function, not a `StarShaclSchema` method; see the [subgraph extraction guide](04f-shacl-subgraph-extraction.ipynb).)

The vocabulary here - `sh:rule`/`sh:TripleRule`/`sh:SPARQLRule`/`sh:condition`, plus the newer `sh:layer`/`sh:runOnce`/`sh:sourceRule`/`sh:RuleSet` - currently lives in the **SHACL 1.2 Inference Rules** editor's draft (`https://w3c.github.io/data-shapes/shacl12-inference-rules/`; not yet published at a `/TR/` URL). It briefly lived inside "SHACL 1.2 SPARQL Extensions" §8 and, before that, in a now-retired standalone "SHACL 1.2 Rules" document - the vocabulary and behavior haven't changed across any of these moves, only the citation has (see [shacl12-gap-matrix.md](../../packages/starlayer/docs/shacl/shacl12-gap-matrix.md) for the full history if you need it). This guide always means the RDF-native `sh:rule` vocabulary `starlayer.shacl` actually executes - not "SPARQL Rule Language" (SRL), a separate human-authoring text syntax for a different, general-purpose Datalog-style rules language that `starlayer.shacl` deliberately doesn't parse (RDF triples are the real interchange form here, the same position this project takes on SHACL's own Compact Syntax).

This guide builds up the feature one concept at a time: basic rule types, execution semantics (fixpoint iteration, layering, rule sets), provenance, RDF 1.2 interplay, and the mutation behavior you need to know before relying on this in real code.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections reuse the running example data from earlier ones.

Graphs here are `StarLayerGraph`, the Oxigraph-backed graph class ("StarLayerGraph" is the working name of what becomes `StarLayerGraph`; see the [Oxigraph wrapper guide](02d-oxigraph-wrapper.ipynb)). `StarLayerGraph` stores its triples in a `pyoxigraph.Store` and parses Turtle 1.2 itself (`format="turtle"`). `StarShaclSchema` accepts `StarLayerGraph` and `StarLayerDataset` directly, and when the data is an `StarLayerGraph` the results come back as `StarLayerGraph` too: `result.inferred_graph` and `result.validation.report_graph` hold pyoxigraph terms and carry only the prefixes of your input.

In [1]:
from rdflib import Namespace
from starlayer.graph.oxgraph import StarLayerGraph
from starlayer.shacl import StarShaclSchema

EX = Namespace("http://example.org/")

## 1. Basic rules: `sh:TripleRule`

A `sh:TripleRule` builds one new triple per matching focus node, from three node-expression templates: `sh:subject`, `sh:predicate`, `sh:object`. `[ shnex:var "focusNode" ]` is the current focus node - the simplest possible template.

Here, every `ex:Person` gets a new `ex:isPerson true` triple. `apply_rules()` returns a `RulesResult` - use `result.inferred_graph` for just the newly-produced triples (see section 11 for exactly what happens to the graph object you passed in, and why `data_graph` itself is left untouched).

In [2]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
    ex:bob a ex:Person .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:PersonRule a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject [ shnex:var "focusNode" ] ; sh:predicate ex:isPerson ; sh:object true ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print(result.inferred_graph.serialize())


@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix ex: <http://example.org/> .
ex:bob ex:isPerson true .
ex:alice ex:isPerson true .



## 2. Node expressions inside rule templates

`sh:subject`/`sh:predicate`/`sh:object` aren't limited to constants like an IRI or a literal - any of the three can be a full SHACL 1.2 node expression (see the separate [Node Expressions guide](04a-shacl-node-expressions.ipynb) for the expression language itself). This is the same mechanism either way; `[ shnex:var "focusNode" ]` is just the simplest possible expression.

**Example 2.1** computes the object: `ex:skillCount` is derived via `shnex:count [ shnex:pathValues ex:skill ]`, the same `shnex:` vocabulary used throughout the Node Expressions guide - not a rules-specific mini-language.


In [3]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:skill ex:python, ex:sparql, ex:rdf .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:S a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject [ shnex:var "focusNode" ] ;
        sh:predicate ex:skillCount ;
        sh:object [ shnex:count [ shnex:pathValues ex:skill ] ] ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print(result.inferred_graph.serialize())


@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix ex: <http://example.org/> .
ex:alice ex:skillCount 3 .



**Example 2.2** goes further and computes the *subject* instead - the rule fires per `ex:Person`, but the inferred triple lands on that person's `ex:manager`, not on the person themselves. This works because a node expression's result, not the focus node, becomes the actual triple subject.

In [4]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:manager ex:carol .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:S a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject [ shnex:pathValues ex:manager ] ;
        sh:predicate ex:hasReports ;
        sh:object true ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print(result.inferred_graph.serialize())
print("ex:carol's inferred triple, even though ex:alice was the target:", (EX.carol, EX.hasReports, None) in result.inferred_graph)


@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix ex: <http://example.org/> .
ex:carol ex:hasReports true .

ex:carol's inferred triple, even though ex:alice was the target: True


## 3. `sh:SPARQLRule`

A `sh:SPARQLRule` builds triples from a `sh:construct` CONSTRUCT query instead of subject/predicate/object templates - useful whenever the derived triple needs real computation (string building, aggregation, joins) that a bare template can't express. `$this` is pre-bound to the focus node inside the query.

Here, `ex:fullName` is built by concatenating `ex:firstName` and `ex:lastName`.

In [5]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:firstName "Alice" ; ex:lastName "Smith" .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:FullNameRule a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:SPARQLRule ;
        sh:construct \"\"\"
          PREFIX ex: <http://example.org/>
          CONSTRUCT { $this ex:fullName ?full }
          WHERE { $this ex:firstName ?f ; ex:lastName ?l . BIND(CONCAT(?f, \" \", ?l) AS ?full) }
        \"\"\" ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print(result.inferred_graph.serialize())


@prefix ex: <http://example.org/> .
ex:alice ex:fullName "Alice Smith" .



## 4. `sh:condition`

`sh:condition` gates which focus nodes a rule fires for: only nodes that conform to the condition shape become focus nodes for that rule. The condition shape needs to actually *constrain* something (`sh:class`, `sh:property`, etc.) - a shape with only `sh:targetClass` trivially conforms for any node when checked this way, since targets only matter for `validate()`'s own target resolution, not an ad hoc conformance check.

Here, only `ex:alice` (an `ex:Adult`) gets the inferred voting-eligibility triple; `ex:bob` (an `ex:Minor`) doesn't.

In [6]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Adult ; ex:name "Alice" .
    ex:bob a ex:Minor ; ex:name "Bob" .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:AdultShape a sh:NodeShape ; sh:class ex:Adult .
    ex:VotingRule a sh:NodeShape ; sh:targetSubjectsOf ex:name ;
      sh:rule [
        a sh:TripleRule ;
        sh:condition ex:AdultShape ;
        sh:subject [ shnex:var "focusNode" ] ; sh:predicate ex:eligibleForVoting ; sh:object true ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print(result.inferred_graph.serialize())


@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix ex: <http://example.org/> .
ex:alice ex:eligibleForVoting true .


## 5. Rules run to a fixpoint

`apply_rules()` re-runs the rules repeatedly until nothing new is produced, as SHACL 1.2 Inference Rules define it - not just once. This matters for anything transitive: one pass only sees one hop, while the fixpoint sees the full closure.

Here, `ex:reach` is built by joining `ex:reach` with itself. A single pass would only derive `a -> c` (one extra hop beyond what is stored); the fixpoint also derives `a -> d` and `b -> d`, since later passes use earlier passes' output. A rule set that never reaches a fixpoint is stopped by `max_iterations` (default 1000), which raises `RuleError`; see section 12.

In [7]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:a ex:reach ex:b .
    ex:b ex:reach ex:c .
    ex:c ex:reach ex:d .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ReachRule a sh:NodeShape ; sh:targetSubjectsOf ex:reach ;
      sh:rule [
        a sh:SPARQLRule ;
        sh:construct \"\"\"
          PREFIX ex: <http://example.org/>
          CONSTRUCT { $this ex:reach ?z }
          WHERE { $this ex:reach ?y . ?y ex:reach ?z }
        \"\"\" ;
      ] .
""", format="turtle")

fixpoint = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print("fixpoint,    ex:a reaches:", sorted(o.value for o in fixpoint.inferred_graph.objects(EX.a, EX.reach)))


fixpoint,    ex:a reaches: ['http://example.org/c', 'http://example.org/d']


## 6. Ordering: `sh:layer` and `sh:runOnce`

By default, each shape's rules reach their own fixpoint independently, shape by shape. `sh:layer` (an integer, default 0) groups rules across *every* shape into ordered layers - all of layer 0 runs (to its own shared fixpoint) before layer 1 starts, so a later layer can depend on an earlier layer's complete output even across shapes.

`sh:runOnce` marks a rule to fire at most once per focus node within its layer, rather than iterating to a fixpoint - essential for a rule that mints something new (like a fresh blank node) on every execution, which would otherwise look "new" forever and hit the iteration cap (section 12).

In [8]:
#layer 1's rule targets whatever layer 0 produced - only works because
#layer 0 has already reached completion by the time layer 1 starts.
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:Layer0Rule a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:TripleRule ; sh:layer 0 ;
        sh:subject [ shnex:var "focusNode" ] ; sh:predicate ex:stage ; sh:object "layer0" ;
      ] .
    ex:Layer1Rule a sh:NodeShape ; sh:targetSubjectsOf ex:stage ;
      sh:rule [
        a sh:TripleRule ; sh:layer 1 ;
        sh:subject [ shnex:var "focusNode" ] ; sh:predicate ex:stage2 ; sh:object "layer1-saw-stage" ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print(result.inferred_graph.serialize())


@prefix ex: <http://example.org/> .
ex:alice ex:stage2 "layer1-saw-stage" ;
	ex:stage "layer0" .



In [9]:
#without sh:runOnce, a rule minting a fresh blank node every execution would
#never converge (each pass looks "new") - sh:runOnce fires it exactly once.
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:MintRule a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:SPARQLRule ; sh:runOnce true ;
        sh:construct \"\"\"
          PREFIX ex: <http://example.org/>
          CONSTRUCT { $this ex:tag [ a ex:Tag ] }
          WHERE { }
        \"\"\" ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print("ex:tag count (expect 1, not an iteration-cap crash):", len(list(result.inferred_graph.objects(EX.alice, EX.tag))))


ex:tag count (expect 1, not an iteration-cap crash): 1


## 7. Rule sets: running only some rules

`sh:RuleSet`/`sh:hasRule`/`sh:includesRuleSet` names a group of rules (with transitive inclusion of other rule sets). `apply_rules(..., rule_set=<IRI>)` restricts execution to that set's closure - every other rule in the shapes graph is simply not run, regardless of which shape it's attached to.

Here, two shapes each contribute one rule; restricting to `ex:GreetingsOnly` runs only `ex:GreetingRule`, leaving `ex:farewell` un-inferred even though `ex:FarewellRule`'s own shape still targets `ex:alice`.

In [10]:
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:R1 a sh:NodeShape ; sh:targetClass ex:Person ; sh:rule ex:GreetingRule .
    ex:GreetingRule a sh:TripleRule ;
      sh:subject [ shnex:var "focusNode" ] ; sh:predicate ex:greeting ; sh:object "hi" .

    ex:R2 a sh:NodeShape ; sh:targetClass ex:Person ; sh:rule ex:FarewellRule .
    ex:FarewellRule a sh:TripleRule ;
      sh:subject [ shnex:var "focusNode" ] ; sh:predicate ex:farewell ; sh:object "bye" .

    ex:GreetingsOnly a sh:RuleSet ; sh:hasRule ex:GreetingRule .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data, rule_set=EX.GreetingsOnly)
print(result.inferred_graph.serialize())


@prefix ex: <http://example.org/> .
ex:alice ex:greeting "hi" .



## 8. Global rules: not attached to any shape

A `sh:SPARQLRule` normally needs a shape's `sh:rule` to reach it - but a bare `sh:SPARQLRule` node with no incoming `sh:rule` edge from any shape still runs, as a *global* rule with no focus node at all (`$this` isn't bound; the query matches against the whole data graph directly). Useful for a fact that isn't naturally about one particular node - here, the symmetric closure of any predicate typed `ex:SymmetricProperty`.

(This global path is deliberately scoped to `sh:SPARQLRule` only - there's no defined meaning for a `sh:TripleRule`'s subject/predicate/object templates with no focus node to evaluate `shnex:var "focusNode"` against.)

In [11]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:siblingOf a ex:SymmetricProperty .
    ex:alice ex:siblingOf ex:bob .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:SymmetricPropertyRule a sh:SPARQLRule ;
      sh:construct \"\"\"
        PREFIX ex: <http://example.org/>
        CONSTRUCT { ?o ?p ?s . }
        WHERE { ?p a ex:SymmetricProperty . ?s ?p ?o . }
      \"\"\" .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print(result.inferred_graph.serialize())


@prefix ex: <http://example.org/> .
ex:bob ex:siblingOf ex:alice .



## 9. Provenance: `sh:sourceRule`

`apply_rules(..., include_source_rule_provenance=True)` attributes every inferred triple to the exact rule that produced it, via an RDF 1.2 reifier: `<< s p o >> {| sh:sourceRule <rule> |}`. Off by default (adds nothing). Useful when more than one rule could plausibly explain the same fact, or for auditing an inference pipeline.

The provenance itself is only materialized after *all* rule execution has fully finished - a rule's own WHERE clause can never see a `sh:sourceRule` triple appear mid-run, even for its own output.

In [12]:
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:R a sh:NodeShape ; sh:targetClass ex:Person ; sh:rule ex:GreetingRule .
    ex:GreetingRule a sh:TripleRule ;
      sh:subject [ shnex:var "focusNode" ] ; sh:predicate ex:greeting ; sh:object "hi" .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data, include_source_rule_provenance=True)
print(result.inferred_graph.serialize(pretty=True, only_used=True))


@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

ex:alice ex:greeting "hi" {| sh:sourceRule ex:GreetingRule |} .



## 10. Rules and RDF 1.2: triple terms

A `sh:SPARQLRule`'s CONSTRUCT can build an RDF 1.2 triple term directly - not just plain triples. Here, alongside the ordinary `ex:knows` fact, the rule also builds a `<<( ... )>>` triple term "witness" of that same fact, executed by Oxigraph, whose SPARQL engine handles RDF 1.2 natively.

In [13]:
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice ex:knows ex:bob .", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:R a sh:NodeShape ; sh:targetSubjectsOf ex:knows ;
      sh:rule [
        a sh:SPARQLRule ;
        sh:construct \"\"\"
          PREFIX ex: <http://example.org/>
          CONSTRUCT { $this ex:knowsWitness <<( $this ex:knows ?o )>> . }
          WHERE { $this ex:knows ?o . }
        \"\"\" ;
      ] .
""", format="turtle")

result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data)
print(result.inferred_graph.serialize())


@prefix ex: <http://example.org/> .
ex:alice ex:knowsWitness <<( ex:alice ex:knows ex:bob )>> .



## 11. `apply_rules()` never mutates its input

`apply_rules()` works against a private working copy of `data_graph` internally - the object you pass in is read once and never mutated, no matter what the rules produce. `result.inferred_graph` is a *new* graph containing only the triples rule execution actually produced; `data_graph` itself still contains exactly what it did before the call.

`apply_rules()` takes `data_graph` as one of StarLayer's own graph classes (`StarLayerGraph`, `StarLayerDataset`, a `pyoxigraph.Store`, or the previous rdflib-based class), and raises `TypeError` for a plain `rdflib.Graph` - load the data into an `StarLayerGraph` first (`StarLayerGraph().extend(g)` accepts rdflib triples). The reason is RDF-1.2 round-tripping, not in-place mutation: only StarLayer's graph classes carry triple terms and other RDF-1.2 constructs correctly through the private copy. The result follows the input: an `StarLayerGraph`, `StarLayerDataset` or `Store` in gives an `StarLayerGraph` out (`result.inferred_graph` and `result.validation.report_graph` hold pyoxigraph terms); the previous rdflib-based class in gives that class out. (`shacl_graph`, as before, is never mutated either - `apply_rules()` always works from a copy of it.)

In [14]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> . @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:R a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [ a sh:TripleRule ; sh:subject [ shnex:var "focusNode" ] ; sh:predicate ex:tag ; sh:object "x" ] .
""", format="turtle")

#an StarLayerGraph - never mutated
sl = StarLayerGraph()
sl.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle")
result = StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=sl)
print("sl itself is untouched - still just the original triple:", list(sl))
print("the new triple only exists in result.inferred_graph:", list(result.inferred_graph))
print("result.inferred_graph is an", type(result.inferred_graph).__name__, "with prefixes", result.inferred_graph.prefixes)
print(result.inferred_graph.serialize(format="turtle"))

#a plain rdflib.Graph - rejected outright, rather than silently returning
#a disconnected copy that only looks like it mutated your original object.
import rdflib
plain = rdflib.Graph()
plain.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle")
try:
    StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=plain)
except TypeError as e:
    print("raised as expected:", e)

sl itself is untouched - still just the original triple: [(<NamedNode value=http://example.org/alice>, <NamedNode value=http://www.w3.org/1999/02/22-rdf-syntax-ns#type>, <NamedNode value=http://example.org/Person>)]
the new triple only exists in result.inferred_graph: [(<NamedNode value=http://example.org/alice>, <NamedNode value=http://example.org/tag>, <Literal value=x datatype=<NamedNode value=http://www.w3.org/2001/XMLSchema#string>>)]
result.inferred_graph is an OxGraph with prefixes {'ex': 'http://example.org/'}
@prefix ex: <http://example.org/> .
ex:alice ex:tag "x" .

raised as expected: apply_rules() requires data_graph to be an OxGraph / OxDataset / pyoxigraph Store or a StarLayerGraph, since rule execution needs RDF-1.2 triple-term support - got Graph. Convert first, e.g. OxGraph() filled with the triples, or StarLayerGraph.from_rdflib(data_graph).


## 12. Error behavior: the iteration cap and unsupported rule types

**The iteration cap is real, not silently truncated.** The engine guards against runaway rule loops with an iteration cap (default 1000): exceeding it raises `starlayer.shacl_engine.rules.RuleError` (a `ValueError`), not a silently-incomplete result. Demonstrated below with the cap lowered through the engine's own `apply_rules(..., max_iterations=3)` to make it reachable quickly (a default of 1000 would take real chains far longer to hit).

**`sh:PropertyRule`/`sh:values`** (a `sh:rule` shorthand - fixed `sh:path`, values from a node expression, no explicit subject/predicate/object template) is **not implemented** - a rule whose type is not `sh:TripleRule` or `sh:SPARQLRule` raises `RuleError` rather than silently no-opping. `sh:TripleRule` and `sh:SPARQLRule` remain the two supported rule types.

In [15]:
from starlayer.shacl_engine import rules
from starlayer.shacl_engine.rules import RuleError

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:a ex:reach ex:b . ex:b ex:reach ex:c . ex:c ex:reach ex:d .
    ex:d ex:reach ex:e . ex:e ex:reach ex:f . ex:f ex:reach ex:g .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ReachRule a sh:NodeShape ; sh:targetSubjectsOf ex:reach ;
      sh:rule [
        a sh:SPARQLRule ;
        sh:construct \"\"\"
          PREFIX ex: <http://example.org/>
          CONSTRUCT { $this ex:reach ?z }
          WHERE { $this ex:reach ?y . ?y ex:reach ?z }
        \"\"\" ;
      ] .
""", format="turtle")

#lower the cap just to make it reachable in this example
try:
    rules.apply_rules(data, shapes, max_iterations=3)
except RuleError as e:
    print("raised as expected:", e)


raised as expected: rules did not reach a fixpoint within 3 iterations


In [16]:
from starlayer.shacl_engine.rules import RuleError

data = StarLayerGraph()
data.parse(data='@prefix ex: <http://example.org/> . ex:alice a ex:Person ; ex:name "Alice" .', format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:R a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [ a sh:PropertyRule ; sh:path ex:fullName ; sh:values ex:name ] .
""", format="turtle")

try:
    StarShaclSchema(shacl_graph=shapes).apply_rules(data_graph=data, meta_shacl=False)
except RuleError as e:
    print("raised as expected:", e)


raised as expected: <BlankNode value=d768f9dd72b2400d7b8e3a856715ed13>: a rule needs an rdf:type that is a subclass of sh:Rule


## Known limitations

- **`sh:tempTriple`** (a triple visible only during rule execution, stripped before the final result) isn't demonstrated here - it's supported for shape-attached rules but not yet wired into the global-rules path (section 8).
- **`sh:expectedPredicate`** (ensuring a `sh:defaultValue`/`sh:values`-derived property exists before a rule runs) isn't demonstrated here - only the `sh:defaultValue` half is implemented; the `sh:values` half depends on the still-unimplemented `sh:PropertyRule`/`sh:values` gap above.
- **Cross-path layering.** The shape-attached layered loop (section 6) and the global-rules layered loop (section 8) are two separate fixpoints, not one unified cross-path fixpoint - a global `sh:runOnce` rule creating instances a shape-attached rule needs as *implicit-class* targets isn't picked up dynamically (targets are computed once, early, before any rule runs).
- **`sh:RuleSet` composition with provenance/layering** works (only the rules actually run get `sh:sourceRule` provenance, and rule sets honor `sh:layer`/`sh:runOnce` normally) but isn't separately demonstrated here.
- **The spec citation is still settling.** As of this writing, the rules vocabulary's home (`SHACL 1.2 Inference Rules`) is an editor's draft with no published `/TR/` snapshot - it moved twice in less than a month before landing here. See [shacl12-gap-matrix.md](../../packages/starlayer/docs/shacl/shacl12-gap-matrix.md)'s "Tracking Upstream Spec Changes" section before citing a specific section number externally.